# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/devotedhruv/flyrank-internships/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

**Author:** Dhurbaraj Singh  
**Track:** Machine Learning · Week 7 (Assignment ML-10)  
**Lane:** Lane 2 — Refresh / Content Opportunity Scoring  
**Data Source:** Built on the FlyRank ML Internship dataset ([https://flyrank.ai](https://flyrank.ai))  

A machine learning score or probability is not an end product. In real-world enterprise SEO, content marketing and editorial teams cannot take action on an abstract float like `P(declining) = 0.78` without understanding **why** the page is flagged and **what specific action** to execute. 

This notebook bridges the gap between predictive modeling and editorial execution by building a production-ready **Content Action Playbook**. We map content archetypes to concrete workflows, attach transparent reason codes, define human-in-the-loop review criteria, delineate strict no-go automation boundaries, formulate monitoring/retraining triggers, and export the queue and visual figures for the capstone research paper.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### A. The Core Decay/Refresh Insight
Standard industry content audits prioritize articles using crude historical traffic volume or publication dates. This heuristic suffers from severe failure modes:
1. **Volume Sorting Selects Evergreen Winners:** High-traffic pages often experience natural minor volume fluctuations without structural decay. Rewriting them wastes valuable writer capacity on content that search engines already trust.
2. **Lagging Realization:** Articles are typically caught only after dropping off Page 1 (positions 1–10), when regaining topical authority is significantly more expensive than proactive maintenance.

Our model identifies pages experiencing or at elevated risk of **algorithmic visibility decay**, weighted by their total recoverable search demand. Rather than treating predictions as a generic binary flag, the pipeline converts model probabilities into an operational priority score (0–100) and maps each asset into an actionable editorial archetype.

### B. Archetype-to-Action Mapping Matrix

| Archetype | Trigger Conditions | Recommended Action | Prescribed Editorial Playbook | Estimated Effort |
| :--- | :--- | :--- | :--- | :--- |
| **Visible Under-Performer** | Page-1 ranking (pos 1–10) + CTR < 25th percentile for tier | `refresh_and_review_ctr` | Retain body content. Audit and rewrite title tags, craft compelling meta descriptions, and implement rich snippet schema to capture lost clicks. | 15–30 mins |
| **High-Interest Engagement Leaker** | Healthy search clicks + scroll/dwell rate < 25th percentile | `refresh_and_review_engagement` | Clicks arrive but readers bounce. Restructure introduction, add interactive table of contents, embed visual media, and improve sub-heading readability. | 1–2 hours |
| **Eroding Cornerstone** | High historical demand + $P(\text{decline}) \ge 0.65$ + position drift | `refresh` | Comprehensive content refresh: update outdated statistics, replace dead sources, add new case studies, and strengthen inbound internal links. | 3–5 hours |
| **Thin Decay Article** | High decay risk + word count < 500 words | `expand_and_refresh` | Competitors have published comprehensive guides. Expand article into a definitive long-form resource covering secondary user intents. | 4–6 hours |
| **Stable Evergreen** | $P(\text{decline}) < 0.40$ + stable trajectory | `monitor` | Zero intervention. Do not touch or modify formatting; avoid breaking existing search equity. | 0 mins |

### C. Transparent Reason Codes
Every queued URL receives human-auditable reason codes explaining the mechanical triggers behind its recommendation:
- `declining_with_demand`: Page is losing visibility on search queries that retain substantial market demand.
- `low_ctr_visible_page`: Page achieves Page-1 rankings but fails to achieve its expected CTR benchmark.
- `low_engagement_visible_page`: Page attracts organic visitors but suffers from sub-benchmark dwell/scroll depth.
- `thin_content_decay`: Word count is under 500 words and actively losing ranking ground.
- `model_decline_risk`: Random Forest calibrated posterior probability of decline $P(\text{decline}) \ge 0.65$.
- `visible_model_opportunity`: High impression footprint ($>1,000$ 90d impressions) coupled with high model decay hazard.
- `ctr_review_candidate` / `engagement_review_candidate`: Specific flag indicating metadata or UX review is required.

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import json

# Resolve repository paths
repo_root = Path("/home/dhruv/flyrank-internships")
queue_path = repo_root / "outputs" / "refresh_queue.csv"
if not queue_path.exists():
    queue_path = repo_root / "outputs" / "refresh_queue_sample.csv"

df_queue = pd.read_csv(queue_path)
print(f"Loaded refresh queue with {len(df_queue):,} rows.")

# 1. Action Distribution Summary
action_counts = df_queue["suggested_action"].value_counts()
action_shares = df_queue["suggested_action"].value_counts(normalize=True) * 100
df_actions = pd.DataFrame({"Count": action_counts, "Share (%)": action_shares.round(1)})
print("\n--- ACTION DISTRIBUTION SUMMARY ---")
print(df_actions.to_string())

# 2. Confidence Tier Distribution
conf_counts = df_queue["confidence"].value_counts()
conf_shares = df_queue["confidence"].value_counts(normalize=True) * 100
df_conf = pd.DataFrame({"Count": conf_counts, "Share (%)": conf_shares.round(1)})
print("\n--- CONFIDENCE TIER SUMMARY ---")
print(df_conf.to_string())

# 3. Display Top 10 Ranked Queue Preview
preview_cols = ["final_rank", "content_id", "final_refresh_score", "best_model_probability", "suggested_action", "confidence", "impressions_90d", "sessions_90d", "final_reason_codes"]
print("\n--- TOP 10 RANKED REFRESH QUEUE PREVIEW ---")
top_10 = df_queue.sort_values("final_refresh_score", ascending=False).head(10)[preview_cols].reset_index(drop=True)
print(top_10[["final_rank", "content_id", "final_refresh_score", "best_model_probability", "suggested_action", "confidence", "impressions_90d", "sessions_90d"]].to_string(index=False))


Loaded refresh queue with 30,000 rows.

--- ACTION DISTRIBUTION SUMMARY ---
                               Count  Share (%)
suggested_action                               
monitor                        13069       43.6
refresh                         8207       27.4
refresh_and_review_ctr          6655       22.2
refresh_and_review_engagement   1987        6.6
expand_and_refresh                82        0.3

--- CONFIDENCE TIER SUMMARY ---
            Count  Share (%)
confidence                  
low         15000       50.0
medium      11424       38.1
high         3576       11.9

--- TOP 10 RANKED REFRESH QUEUE PREVIEW ---
 final_rank           content_id  final_refresh_score  best_model_probability       suggested_action confidence  impressions_90d  sessions_90d
          1 content_1f080331fa2b            81.928467                0.786247 refresh_and_review_ctr       high            12834            66
          2 content_6aa43079fb0c            81.728449                0.792117 r

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### A. Intended Audience & Operational Cadence
- **Target Users:** Content Marketing Strategists, SEO Specialists, and Managing Editors.
- **Primary Use Case:** Monthly or bi-weekly **editorial sprint planning**.
- **Operational Workflow:** Rather than manually browsing hundreds of published URLs in Google Search Console, editors filter the queue for `confidence == 'high'` and sort by `final_refresh_score DESC`. Selecting the top 30–50 items establishes an evidence-backed sprint backlog in under 15 minutes.

### B. Cost/Value Thinking & Resource Allocation
Editorial bandwidth is expensive and strictly finite. A full-time content editor typically produces or refreshes approximately 25 to 40 URLs per month (~160 hours total).

The playbook incorporates cost-aware triage:
1. **Quick-Win Triage (`refresh_and_review_ctr`):** Metadata updates require only ~20 minutes of copywriting effort per URL. Reviewing 20 high-impression CTR-decay pages takes ~7 hours but can recapture thousands of search clicks.
2. **Deep-Dive Triage (`refresh` & `expand_and_refresh`):** Comprehensive content rewrites consume 4–6 hours per article. The model reserves this high-cost investment exclusively for high-demand URLs with verified algorithmic decay, preventing wasted effort on healthy assets.

### C. Methodological Boundaries & Limitations
In accordance with the `writing-honest-claims` framework:
- **Observational, Not Causal:** The model identifies statistical associations between historical signals and subsequent traffic decay. It **cannot prove** that updating an article will causally restore lost rankings; causal claims require prospective randomized A/B testing.
- **No Google Algorithm Reverse-Engineering:** This system models empirical visibility patterns within an enterprise portfolio; it makes no claims of uncovering Google's proprietary search algorithms.
- **External Demand Shocks:** Macroeconomic disruptions, seasonal holiday shifts, or broad market trends can decrease search volume externally. These appear as content decay in behavioral telemetry even when content quality remains pristine.
- **Decision-Support Role:** This tool acts as an intelligent prioritization assistant, never an autonomous decision-maker.

In [2]:
# Cost-Benefit Simulation for a 40-URL Editorial Sprint
sprint_candidates = df_queue[df_queue["confidence"] == "high"].sort_values("final_refresh_score", ascending=False).head(40).copy()

effort_mapping = {
    "refresh_and_review_ctr": 0.35,        # 20 mins
    "refresh_and_review_engagement": 1.5,   # 1.5 hours
    "refresh": 3.5,                         # 3.5 hours
    "expand_and_refresh": 5.0,              # 5 hours
    "monitor": 0.0
}

sprint_candidates["estimated_hours"] = sprint_candidates["suggested_action"].map(effort_mapping)
total_hours = sprint_candidates["estimated_hours"].sum()
total_impressions_at_risk = sprint_candidates["impressions_90d"].sum()
action_breakdown = sprint_candidates.groupby("suggested_action").agg(
    urls_queued=("content_id", "count"),
    avg_score=("final_refresh_score", "mean"),
    total_hours=("estimated_hours", "sum"),
    total_impressions=("impressions_90d", "sum")
).round(1)

print("=== SIMULATED 40-URL SPRINT ALLOCATION ===")
print(f"Total Editorial Hours Required: {total_hours:.1f} hours (~{total_hours/160:.1%} of 1 FTE editor)")
print(f"Total Search Impressions at Risk: {total_impressions_at_risk:,} impressions")
print("\nBreakdown by Recommended Action:")
print(action_breakdown.to_string())


=== SIMULATED 40-URL SPRINT ALLOCATION ===
Total Editorial Hours Required: 31.5 hours (~19.7% of 1 FTE editor)
Total Search Impressions at Risk: 386,326 impressions

Breakdown by Recommended Action:
                               urls_queued  avg_score  total_hours  total_impressions
suggested_action                                                                     
refresh                                  3       78.9         10.5              36965
refresh_and_review_ctr                  30       79.5         10.5             263085
refresh_and_review_engagement            7       79.0         10.5              86276


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### A. The 4-Step Human Review Checklist
Before executing changes on any URL recommended by the queue, an editor must verify:
1. **Search Intent Verification:** Did user search intent evolve? (e.g., Has an informational query shifted into a transactional tool query or video intent? If so, adding text won't help; a calculator or visual asset is required).
2. **SERP Real Estate Inspection:** Did search engines introduce an AI Overview, video carousel, or shopping pack at the top of the search results? If an AI Overview occupies the top 800 pixels, CTR drops are driven by layout changes rather than content decay.
3. **Business & Conversion Relevance:** Does this URL actually drive revenue, newsletter signups, or qualified leads? If a decaying page generates zero business value, writer hours are better spent elsewhere.
4. **Factual & Technical Audit:** Confirm that product details, technical screenshots, pricing references, and cited studies are accurate and up-to-date.

### B. The Strict No-Go List (Never Automate)
- ❌ **No Autonomous Bulk AI Rewriting:** Never deploy LLMs to automatically rewrite and publish content directly to production CMS. AI models hallucinate facts, erode brand tone, and risk search engine quality penalties.
- ❌ **No Automated 301 Redirects or URL Pruning:** Never delete or redirect pages automatically. Pruning can break internal link equity or kill long-tail query rankings across unmonitored terms.
- ❌ **Never Change Established URL Slugs:** Modifying existing URL paths risks severing external backlinks and losing established domain equity.
- ❌ **Zero Unsupervised Edits on YMYL Content:** Health, medical, legal, and financial (Your Money Your Life) content must be authored and reviewed exclusively by verified subject matter experts.

In [3]:
# Automated Pre-Flight Safety Verification Gate
def safety_preflight_audit(url_record):
    """
    Evaluates a candidate record against the no-go safety criteria before
    allowing editorial work to be scheduled.
    """
    flags = []
    # 1. YMYL intent check
    if url_record.get("main_intent") in ["financial", "medical", "legal"]:
        flags.append("BLOCKED_YMYL: Requires certified SME review before touching.")
    # 2. Extreme low volume check
    if url_record.get("impressions_90d", 0) < 50:
        flags.append("LOW_VOLUME_WARNING: Insufficient search demand to justify rewriting.")
    # 3. Slug alteration protection
    if url_record.get("suggested_action") == "prune_and_redirect":
        flags.append("NO_GO_VIOLATION: Automated 301 redirects are strictly prohibited.")
        
    return "APPROVED" if not flags else " | ".join(flags)

# Test pre-flight gate on sample candidates
sample_row = df_queue.iloc[0].to_dict()
status = safety_preflight_audit(sample_row)
print(f"Pre-Flight Safety Audit Result for Top Candidate ({sample_row['content_id']}): {status}")


Pre-Flight Safety Audit Result for Top Candidate (content_1f080331fa2b): APPROVED


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Content ranking environments are non-stationary: seasonal cycles, industry shifts, and search algorithm updates cause model performance to degrade over time. We establish explicit **drift monitoring signals** and **retraining triggers**.

### A. Key Stagnation & Drift Signals
1. **Precision@50 Degradation:** Editorial teams track sprint verification rates. If the true positive decline rate in the top 50 audited URLs drops below **50%** across two consecutive sprints, model discrimination has decayed.
2. **Distributional Covariate Shift:** Substantial shifts in input feature distributions (e.g., population mean CTR dropping by >20% following an AI Overview rollout) indicate input feature drift.
3. **Ratio of Low-Confidence Recommendations:** If the proportion of high-confidence predictions drops below **10%** of the inventory, the model's posterior probability calibration is weakening.

### B. Operational Retraining Cadence
- **Quarterly Scheduled Retraining:** Retrain models every 90 days using the most recent 90-day observation window.
- **Event-Driven Retraining:** Retrain within 14 days following confirmed Google Search Core Updates or major AI search interface overhauls.

In [4]:
# Health Check & Drift Detection Monitor
def evaluate_model_health(df):
    high_conf_pct = (df["confidence"] == "high").mean() * 100
    actionable_pct = (df["suggested_action"] != "monitor").mean() * 100
    mean_score = df["final_refresh_score"].mean()
    
    status = "HEALTHY"
    warnings = []
    if high_conf_pct < 10.0:
        status = "WARNING"
        warnings.append("High-confidence candidate share below 10% threshold.")
    if actionable_pct < 20.0:
        status = "WARNING"
        warnings.append("Actionable item share unusually low (<20%).")
        
    return {
        "status": status,
        "high_confidence_share": f"{high_conf_pct:.1f}%",
        "actionable_share": f"{actionable_pct:.1f}%",
        "mean_queue_score": f"{mean_score:.1f}",
        "alerts": warnings if warnings else ["All telemetry metrics within nominal bounds."]
    }

health_report = evaluate_model_health(df_queue)
print("=== MODEL TELEMETRY HEALTH & DRIFT AUDIT ===")
for k, v in health_report.items():
    print(f"{k.replace('_', ' ').title()}: {v}")


=== MODEL TELEMETRY HEALTH & DRIFT AUDIT ===
Status: HEALTHY
High Confidence Share: 11.9%
Actionable Share: 56.4%
Mean Queue Score: 49.8
Alerts: ['All telemetry metrics within nominal bounds.']


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The capstone research paper requires committed figures and receipts. In this step, we ensure all figures are generated and stored in `work/figures/`, and metrics receipts and queue summaries are preserved in `work/outputs/`.

In [5]:
import shutil

figures_dir = repo_root / "work" / "figures"
outputs_dir = repo_root / "work" / "outputs"
figures_dir.mkdir(parents=True, exist_ok=True)
outputs_dir.mkdir(parents=True, exist_ok=True)

# 1. Export sample queue for reporting
sample_out_path = outputs_dir / "refresh_queue_sample.csv"
df_queue.head(500).to_csv(sample_out_path, index=False)
print(f"Exported sample queue (500 rows) to: {sample_out_path}")

# 2. Copy and verify chart figures
source_charts = repo_root / "outputs" / "charts"
if source_charts.exists():
    for chart_file in source_charts.glob("*.svg"):
        dest_file = figures_dir / chart_file.name
        shutil.copy(chart_file, dest_file)
        print(f"Verified figure: {dest_file.name} ({dest_file.stat().st_size:,} bytes)")

# 3. Export Playbook Metadata Summary JSON
playbook_summary = {
    "lane": "Refresh / Content Opportunity Scoring",
    "total_urls_scored": len(df_queue),
    "high_confidence_candidates": int((df_queue["confidence"] == "high").sum()),
    "action_distribution": action_counts.to_dict(),
    "confidence_distribution": conf_counts.to_dict(),
    "top_queue_candidate": {
        "content_id": str(df_queue.iloc[0]["content_id"]),
        "final_score": float(df_queue.iloc[0]["final_refresh_score"]),
        "action": str(df_queue.iloc[0]["suggested_action"]),
        "reasons": str(df_queue.iloc[0]["final_reason_codes"])
    }
}

summary_json_path = outputs_dir / "playbook_summary.json"
with open(summary_json_path, "w") as f:
    json.dump(playbook_summary, f, indent=2)
print(f"Exported playbook summary JSON to: {summary_json_path}")


Exported sample queue (500 rows) to: /home/dhruv/flyrank-internships/work/outputs/refresh_queue_sample.csv
Verified figure: top_feature_importance.svg (3,607 bytes)
Verified figure: trend_distribution.svg (1,631 bytes)
Verified figure: action_mix.svg (1,680 bytes)
Verified figure: confidence_mix.svg (1,088 bytes)
Verified figure: top_reason_codes.svg (3,433 bytes)
Exported playbook summary JSON to: /home/dhruv/flyrank-internships/work/outputs/playbook_summary.json


## 6. Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

## 7. ML-12 Tell the Story: 5-Minute Showcase Demo Outline & Shareable Cuts

### A. 5-Minute Showcase Demo Outline
- **[0:00 – 1:00] Question & Problem:** How can search editorial teams identify decaying content across enterprise catalogs within strict 20–50 URL sprint budgets without guessing?
- **[1:00 – 2:00] Method & Leakage Defense:** 30,000 URLs across 32 clients; 52 signals; quarantined 8 outcome-window columns; evaluated strictly on held-out client domains.
- **[2:00 – 3:00] One Key Visual:** Display `Figure 1: Top Feature Importances` — index appearance frequency (`days_with_impressions`, 16.06%) and content length moat dominate stability.
- **[3:00 – 4:00] One Honest Result:** Random Forest achieved **Precision@50 of 0.700** and **ROC AUC of 0.747** vs heuristic baseline Precision@50 of 0.240 (2.92x lift). Note that this is observational decision-support, not causal algorithmic proof.
- **[4:00 – 5:00] One Concrete Recommendation:** Deploy the reason-coded Action Playbook: leave 43.6% in `monitor`, and prioritize the top-50 high-confidence URLs (`refresh_and_review_ctr`, `refresh`, `expand_and_refresh`) to recover ~35 high-impact assets per sprint.

### B. Shareable Cut 1: Methodology Social Post (LinkedIn / X)
> "Can machine learning predict which published articles are decaying in Google Search before traffic falls off a cliff?
>
> In my FlyRank Search Intelligence Capstone, I trained a Random Forest model on 30,000 URLs across 32 enterprise client domains using a strict client-holdout split to prevent leakage.
>
> Key findings:
> • Precision@50 rose from 24% (heuristic baseline) to 70% (model), a 2.92x lift in high-value refresh candidates.
> • Strongest stability signal: index appearance consistency (`days_with_impressions`). When continuous appearance drops to sporadic weekly impressions, algorithmic erosion has already started.
> • High search volume queries decay at virtually the same rate as long-tail queries (<1.1% importance).
>
> Live research paper: https://devotedhruv.github.io/flyrank-internships/
> Code & Artifacts: https://github.com/devotedhruv/flyrank-internships
>
> Built on the FlyRank ML Internship dataset (https://flyrank.ai)"

### C. Shareable Cut 2: 3-Sentence Employer-Facing Summary
> "I engineered an end-to-end Machine Learning search intelligence pipeline on 30,000 multi-client enterprise search records from FlyRank's telemetry that predicts organic visibility decay using 52 behavioral and technical signals. Evaluated under a grouped client-holdout cross-validation split with strict leakage quarantines, my Random Forest model achieved an ROC AUC of 0.747 and boosted top-50 recommendation precision from 24% to 70% (a 2.92x lift) over standard industry heuristics. I translated these predictions into an operational, reason-coded editorial action playbook deployed as an open-access research paper and reproducible GitHub repository."
